# FeatureSelector Walkthrough

`FeatureSelector` is a self-contained bootstrap feature selection tool. It takes raw
`(X, y)` — no base model, no pre-computed residuals — and evaluates each feature
independently against a permuted-null baseline.

## Workflow

1. Load data and prepare a feature matrix and target vector.
2. Fit `FeatureSelector` on `(X, y)`.
3. Inspect `summary_` and `selected_features_`.
4. Call `.find_interactions()` to detect which selected feature pairs show interaction
   effects beyond their additive main effects.
5. Visualise with `plot_selected_features()`, `plot_feature_response()`,
   `plot_feature_correlations()`, and `plot_interactions()`.
6. Export a self-contained HTML report with `to_html()`.

## What FeatureSelector measures

For each feature, across many bootstrap splits:

- Fits a depth-1 ensemble model (`XGBoost` or `RandomForest`) predicting `y` from
  that feature alone.
- Fits the same model on a **permuted version** of the feature (null baseline).
- Records two scores per split:
  - **R²** — explained variance.
  - **Robust metric** — Spearman rank correlation (regression) or Gini index = 2·AUC − 1 (binary).
- A feature is **selected** when `null_beat_rate ≥ 0.80` (beats null on both metrics)
  and `positive_score_rate ≥ 0.80` (positive score in ≥ 80% of splits).

`find_interactions()` then runs the bivariate depth-1 vs depth-2 bootstrap on
`selected_features_` to identify which pairs predict `y` better jointly than additively.

## Setup

In [31]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for path in [PROJECT_ROOT / 'src', PROJECT_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT

WindowsPath('C:/Users/amsac/Desktop/repos/modeling-tools')

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

matplotlib.use('Agg')
%matplotlib inline

from pe_tools.signal_finder import FeatureSelector

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

## Load data

We use a 20 000-row stratified sample of the UCI **Default of Credit Card Clients** dataset.
An `age_band` categorical column is added to exercise categorical feature handling.

`FeatureSelector` takes the raw features and target directly — no base model or residuals needed.

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'default_of_credit_card_clients.csv'
credit = pd.read_csv(DATA_PATH)
target_col = 'default_next_month'
feature_cols = [c for c in credit.columns if c != target_col]

analysis_frame = train_test_split(
    credit,
    train_size=20_000,
    stratify=credit[target_col],
    random_state=42,
)[0].sort_index()

X = analysis_frame[feature_cols].copy()
X['age_band'] = pd.cut(
    X['age'], bins=[20, 30, 40, 50, 60, 80], include_lowest=True
).astype(str)
y = analysis_frame[target_col].astype(float)

positive_rate = float(y.mean())
print(f'Rows: {len(analysis_frame):,}   Positive rate: {positive_rate:.3f}')
print(f'Features: {X.shape[1]}')

## Fit FeatureSelector

`FeatureSelector` evaluates each feature independently against a permuted-null baseline.
`n_bootstraps=30` is used here for notebook speed; use 50+ in production.
Set `verbose=True` to see per-feature progress during the univariate loop.

In [ ]:
selector = FeatureSelector(
    n_bootstraps=30,
    split_strategy='stratified_bootstrap',  # preserves class balance for binary target
    model_type='xgboost',
    max_depth=1,
    null_beat_rate_threshold=0.80,
    positive_score_rate_threshold=0.80,
    random_state=42,
    verbose=True,
)
selector.fit(X, y)

print(f'\nTask detected: {"binary" if selector._is_binary else "regression"}')
print(f'Features evaluated: {len(selector._candidate_features)}')
print(f'Features selected:  {len(selector.selected_features_)}')
print(f'\nSelected features: {selector.selected_features_}')

## Inspect results

`summary_` has one row per feature, sorted by `null_beat_rate` descending.

| Column | Meaning |
|---|---|
| `mean_r2` | Mean R² across bootstrap splits. |
| `mean_robust_metric` | Mean Gini = 2·AUC − 1 (binary) or Spearman ρ (regression). |
| `std_robust_metric` | Standard deviation of the robust metric across splits — measures stability. |
| `null_beat_rate` | Fraction of splits where the real feature beat its permuted null on **both** metrics. |
| `positive_score_rate` | Fraction of splits with R² > 0 and robust metric > 0. |
| `selected` | `True` when `null_beat_rate ≥ 0.80` AND `positive_score_rate ≥ 0.80`. |

In [ ]:
display(
    selector.summary_[
        ['feature', 'dtype', 'mean_r2', 'mean_robust_metric', 'std_robust_metric',
         'null_beat_rate', 'positive_score_rate', 'selected']
    ].head(20)
)

## Visualise feature selection

`plot_selected_features()` returns a two-panel figure:

- **Left**: mean robust metric (Gini or Spearman) with standard deviation error bars.
  Green bars passed both selection thresholds; grey bars did not.
- **Right**: histogram of null beat rates across all features, with the selection
  threshold annotated as a dashed red line.

In [ ]:
fig = selector.plot_selected_features(top_n=24)
plt.show()
plt.close(fig)

## Feature response and correlation

`plot_feature_response()` shows the univariate relationship between each selected
feature and the target `y`:

- **Continuous features**: quantile-binned mean(y) with observation counts below.
- **Categorical features**: mean(y) per category, sorted descending.

`plot_feature_correlations()` shows pairwise Spearman correlations among selected
features (lower triangle only). Pairs with |rho| < `correlation_threshold` are shown
as blank. Useful for spotting redundant features before interaction search.

In [ ]:
fig = selector.plot_feature_response(top_n=12)
plt.show()
plt.close(fig)

if not selector.feature_correlation_.empty:
    fig = selector.plot_feature_correlations()
    plt.show()
    plt.close(fig)

## Find interactions

`find_interactions()` runs a bivariate bootstrap on `selected_features_`. For each pair
it fits a depth-1 (additive) and depth-2 (interaction) tree model per split and records
the lift R²(depth-2) − R²(depth-1). A permuted-null version of `feature_b` is evaluated
in parallel to establish the chance baseline.

`interaction_summary_` has one row per pair, sorted by `mean_interaction_lift` descending.

| Column | Meaning |
|---|---|
| `mean_interaction_lift` | Mean R²(depth-2) − R²(depth-1) across bootstraps. Positive = joint prediction better than additive. |
| `median_interaction_lift` | Median lift; less sensitive to outlier splits. |
| `positive_lift_rate` | Fraction of splits where lift > 0.001 (materiality threshold). |
| `interaction_null_beat_rate` | Fraction of splits where real lift > permuted-null lift. |
| `mean_depth2_r2` | Mean OOF R² of the depth-2 (interaction) model. |
| `mean_depth1_r2` | Mean OOF R² of the depth-1 (additive) model. |

In [ ]:
selector.find_interactions(top_n=10)

print(f'Bootstrap results shape: {selector.interaction_bootstrap_results_.shape}')
print(f'Summary shape:           {selector.interaction_summary_.shape}\n')
display(selector.interaction_summary_)

## Credibility filter

Require `positive_lift_rate > 0.40` and `interaction_null_beat_rate > 0.40` to surface
pairs that consistently show real interaction lift rather than noise. Tighten thresholds
(e.g., 0.65 / 0.75) for production feature engineering decisions.

In [ ]:
POSITIVE_LIFT_THRESHOLD = 0.40
NULL_BEAT_THRESHOLD_INTERACTION = 0.40

credible_pairs = selector.interaction_summary_.loc[
    (selector.interaction_summary_['positive_lift_rate'] > POSITIVE_LIFT_THRESHOLD)
    & (selector.interaction_summary_['interaction_null_beat_rate'] > NULL_BEAT_THRESHOLD_INTERACTION)
].copy()

print(f'Pairs passing credibility filter: {len(credible_pairs)} of {len(selector.interaction_summary_)}')
display(
    credible_pairs[[
        'feature_1', 'feature_2', 'rank',
        'mean_interaction_lift', 'positive_lift_rate', 'interaction_null_beat_rate',
    ]]
)

## Per-bootstrap results

`interaction_bootstrap_results_` has one row per (feature_pair × bootstrap split).
Use it to inspect the distribution of lift and null lift for a specific pair.

In [ ]:
top_pair = selector.interaction_summary_.iloc[0]
f1, f2 = top_pair['feature_1'], top_pair['feature_2']

pair_runs = selector.interaction_bootstrap_results_.loc[
    (selector.interaction_bootstrap_results_['feature_1'] == f1)
    & (selector.interaction_bootstrap_results_['feature_2'] == f2)
].copy()

print(f'Top pair: {f1} × {f2}   (rank {int(top_pair["rank"])})')
print(f'  mean lift:                {pair_runs["interaction_lift"].mean():.5f}')
print(f'  mean null lift:           {pair_runs["null_lift"].mean():.5f}')
print(f'  interaction_null_beat_rate: {pair_runs["beats_null"].mean():.2f}')
print()
display(pair_runs.head(10))

## Diagnostic plots

`plot_interactions()` returns one figure per top-N ranked pair. Each figure has
**three panels**:

- **Left — Interaction effect**: hexbin plot colored by the mean difference between
  depth-2 and depth-1 predictions (RdBu_r palette). Red = positive interaction;
  blue = negative. Layout adapts to feature type:
  - Continuous × continuous → hexbin colored by mean net effect.
  - Continuous × categorical → conditional mean-effect curves, one line per category.
  - Categorical × categorical → bubble chart; color = mean net effect, size = count.

- **Centre — Conditional mean of y**: the same feature space colored by mean target value
  (YlOrRd palette). Compare with the left panel to see where interaction effects
  align with actual target outcomes.

- **Right — Lift distribution**: bootstrap interaction lift (blue) vs permuted-null
  lift (grey), with `interaction_null_beat_rate` annotated.

In [ ]:
figures = selector.plot_interactions(top_n=5)

for pair_key, fig in figures.items():
    feat_a, feat_b = pair_key.split('__x__')
    row = selector.interaction_summary_.loc[
        ((selector.interaction_summary_['feature_1'] == feat_a) &
         (selector.interaction_summary_['feature_2'] == feat_b))
        | ((selector.interaction_summary_['feature_1'] == feat_b) &
           (selector.interaction_summary_['feature_2'] == feat_a))
    ].iloc[0]
    print(
        f'Pair {int(row["rank"])}: {feat_a} × {feat_b}   '
        f'lift={row["mean_interaction_lift"]:.5f}   '
        f'null_beat_rate={row["interaction_null_beat_rate"]:.2f}'
    )
    display(fig)
    plt.close(fig)

In [ ]:
# Export a self-contained HTML report
report_path = PROJECT_ROOT / "notebooks" / "outputs" / "feature_selector_report.html"
report_path.parent.mkdir(parents=True, exist_ok=True)

html = selector.to_html(
    path=str(report_path),
    title="Credit Default — Feature Selection Report",
    top_n=5,
)
print(f"Report written: {report_path} ({len(html):,} chars)")

## Interpreting results

**Reading `summary_`:**

- `null_beat_rate` is the primary credibility signal. A feature at 0.95 reliably beats
  its permuted baseline; one at 0.50 is indistinguishable from chance.
- `mean_robust_metric` (Gini or Spearman) captures the magnitude. A high `null_beat_rate`
  with a very low robust metric means the feature is consistently predictive but weakly so.
- Features near the selection thresholds (e.g., `null_beat_rate ≈ 0.80`) warrant extra
  scrutiny — increase `n_bootstraps` to 50+ to confirm stability.

**Reading `interaction_summary_`:**

- `interaction_null_beat_rate ≥ 0.75` and `positive_lift_rate ≥ 0.65` with
  `mean_interaction_lift > 0.002` is a reasonable threshold for a pair worth engineering.
- A pair where `mean_depth1_r2` is already near zero may show a spuriously large
  *relative* lift. Always check the absolute depth-2 R² value.

**Next steps after identifying interactions:**

1. Engineer an explicit interaction feature (e.g., `pay_0 * bill_amt1`, or a ratio).
2. Verify the engineered feature passes `FeatureSelector` univariate selection on its own.
3. Use `refinement_enabled=True` to confirm it adds marginal value beyond the individual
   selected features.

**Computational cost guidance:**

| Candidate features | Pairs | Bootstraps | Total model fits |
|---|---|---|---|
| 5  | 10  | 50 | 2 000  |
| 10 | 45  | 50 | 9 000  |
| 15 | 105 | 50 | 21 000 |

Each fit trains 4 models (depth-1 real, depth-2 real, depth-1 null, depth-2 null).
Use `model_params` to reduce `n_estimators` when exploring, then increase for a final run.

## Result attributes at a glance

| Attribute | Populated by | Content |
|---|---|---|
| `summary_` | `.fit()` | One row per feature: scores, null beat rate, selected flag |
| `bootstrap_results_` | `.fit()` | One row per feature x split: raw R2 and robust metric |
| `selected_features_` | `.fit()` | List of features passing both thresholds |
| `feature_correlation_` | `.fit()` | Spearman rho matrix among selected features (NaN where |rho| < threshold) |
| `interaction_summary_` | `.find_interactions()` | One row per pair: lift, null beat rate, rank |
| `interaction_bootstrap_results_` | `.find_interactions()` | One row per pair x split: raw depth-1 and depth-2 R2 |